# 🧠 NLA Hallucination Detection Pipeline (v2 — Raw Vector Protocol)
**Senior Project — SIIT Thammasat University**  
**Model:** Qwen2.5-7B-Instruct | **Dataset:** HaluEval QA + In-dist validation | **NLA Layer:** 20

| Phase | Description | VRAM |
|-------|-------------|------|
| 0 — Setup | Install deps + download models | Low |
| Session 1A | Extract Layer 20 activations — HaluEval (Qwen only) | ~14 GB |
| Session 1B | Extract Layer 20 activations — In-dist (WildChat + Ultra-FineWeb) | ~14 GB |
| Session 2 | NLA Verbalization (AV) + AR Scoring — both datasets | ~40 GB |

> ⚠️ **Session 1 and Session 2 must run in separate kernels** (Qwen + SGLang > 48 GB combined)

### 🔄 Changes from v1 (per Qwen NLA validation protocol)
1. **No L2-normalization at rest** — vectors saved as raw float32; scaling happens at inference via `injection_scale`
2. **MIN_POSITION = 50** — only sample tokens at index ≥ 50 (early tokens = attention sinks / immature features)
3. **Greedy decoding** at AV (deterministic, temperature = 0)
4. **New Session 1B** — in-distribution validation set (50% WildChat-1M + 50% Ultra-FineWeb) to reproduce the 0.752 fve_nrm baseline and prove the pipeline is correct

## 🔧 0. Environment Check

In [1]:
import torch

print(f'PyTorch:    {torch.__version__}')
print(f'CUDA:       {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU:        {props.name}')
    print(f'Total VRAM: {props.total_memory / 1e9:.1f} GB')
    print(f'Used  VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
!nvidia-smi | grep MiB

PyTorch:    2.9.1+cu128
CUDA:       True
GPU:        NVIDIA A40
Total VRAM: 47.7 GB
Used  VRAM: 0.0 GB
|  0%   33C    P8             33W /  300W |       3MiB /  46068MiB |      0%      Default |


## 📦 1. Install Dependencies

In [2]:
!pip install --break-system-packages huggingface_hub hf_transfer accelerate datasets scikit-learn tqdm -q
!pip install --break-system-packages sglang==0.5.6 -q
!cd /workspace/natural_language_autoencoders && pip install --break-system-packages -e . -q
print('✅ All dependencies installed!')

✅ All dependencies installed!


## 📥 2. Download Models
> ⚠️ `allenai/WildChat-1M` is a **gated dataset** — accept the license on HuggingFace and login with `huggingface-cli login` (or set `HF_TOKEN`) before Session 1B.

In [3]:
import os
os.environ.update({
    'HF_HOME': '/workspace/hf_cache',
    'TRANSFORMERS_CACHE': '/workspace/hf_cache',
    'HF_HUB_ENABLE_HF_TRANSFER': '1',   # Rust downloader — fixes per-connection throughput first
})

from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import snapshot_download

MODELS = {
    'Qwen/Qwen2.5-7B-Instruct':    '/workspace/models/qwen2.5-7b-instruct',
    'kitft/nla-qwen2.5-7b-L20-av': '/workspace/models/nla-av',
    'kitft/nla-qwen2.5-7b-L20-ar': '/workspace/models/nla-ar',
}

def download_one(repo_id, local_dir):
    if os.path.exists(local_dir) and os.listdir(local_dir):
        return repo_id, local_dir, 'skipped'
    snapshot_download(repo_id=repo_id, local_dir=local_dir)
    return repo_id, local_dir, 'downloaded'

errors = {}
with ThreadPoolExecutor(max_workers=len(MODELS)) as pool:
    futures = {pool.submit(download_one, repo_id, local_dir): repo_id
               for repo_id, local_dir in MODELS.items()}
    for future in as_completed(futures):
        repo_id = futures[future]
        try:
            _, local_dir, status = future.result()
            icon = '⏩' if status == 'skipped' else '✅'
            print(f'{icon} {status.capitalize()}: {repo_id} -> {local_dir}')
        except Exception as e:
            errors[repo_id] = e
            print(f'❌ Failed: {repo_id} -> {e}')

if errors:
    raise RuntimeError(f'{len(errors)} model download(s) failed: {list(errors)}')

# Verify
print()
for path in MODELS.values():
    n = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f'  {"✅" if n > 0 else "❌"} {path.split("/")[-1]:35s} ({n} files)')
print('\n✅ All models ready!')

⏩ Skipped: kitft/nla-qwen2.5-7b-L20-ar -> /workspace/models/nla-ar
⏩ Skipped: Qwen/Qwen2.5-7B-Instruct -> /workspace/models/qwen2.5-7b-instruct
⏩ Skipped: kitft/nla-qwen2.5-7b-L20-av -> /workspace/models/nla-av

  ✅ qwen2.5-7b-instruct                 (15 files)
  ✅ nla-av                              (19 files)
  ✅ nla-ar                              (19 files)

✅ All models ready!


---
## 🧠 SESSION 1A — Extract Layer 20 Activations (HaluEval, RAW)

**Protocol changes vs v1:**
- ❌ No L2-normalization — save **raw residual stream vectors** (float32)
- ✅ Record last-token position; flag samples where position < `MIN_POSITION` (50)
- Hook on `model.model.layers[20]` output = residual stream **after** block 20, **before** final norm ✅

> **After running all Session 1 cells: restart the kernel before Session 2!**

In [4]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

# ─── Config ─────────────────────────────────────
MODEL_PATH   = '/workspace/models/qwen2.5-7b-instruct'
LAYER_IDX    = 20
N_SAMPLES    = 200
MIN_POSITION = 50          # protocol: tokens at index >= 50 only
SAVE_DIR     = '/workspace'

# ─── Load Qwen ──────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model.eval()
print(f'✅ Qwen loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ─── Load HaluEval ──────────────────────────────
dataset = load_dataset('pminervini/HaluEval', 'qa_samples', split='data')
samples = dataset.select(range(N_SAMPLES))
print(f'✅ HaluEval: {len(samples)} samples')

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Qwen loaded | VRAM: 15.2 GB
✅ HaluEval: 200 samples


In [5]:
# ─── Shared extraction utilities (used by 1A and 1B) ──
_captured = {}

def hook_fn(module, input, output):
    hidden = output[0] if isinstance(output, tuple) else output
    # Keep FULL sequence hidden states; raw, no normalization
    _captured['h'] = hidden.detach().float().cpu()   # [1, seq, hidden]

hook = model.model.layers[LAYER_IDX].register_forward_hook(hook_fn)

@torch.no_grad()
def extract_at_position(text: str, position: int = -1, max_length: int = 1024):
    """Run forward pass, return (raw_vector[hidden], token_position, seq_len)."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=max_length).to('cuda')
    model(**inputs)
    seq_len = _captured['h'].shape[1]
    pos = seq_len - 1 if position == -1 else min(position, seq_len - 1)
    return _captured['h'][0, pos, :].clone(), pos, seq_len

In [6]:
activations_list, labels_list, positions_list = [], [], []

for sample in tqdm(samples, desc=f'HaluEval L{LAYER_IDX} extraction (raw)'):
    prompt = f"Question: {sample['question']}\nAnswer: {sample['answer']}"
    vec, pos, seq_len = extract_at_position(prompt, position=-1, max_length=512)
    activations_list.append(vec)
    positions_list.append(pos)
    labels_list.append(1 if sample['hallucination'] == 'yes' else 0)

activations_np = torch.stack(activations_list).numpy().astype(np.float32)  # RAW
labels_np      = np.array(labels_list)
positions_np   = np.array(positions_list)

np.save(f'{SAVE_DIR}/activations_L20_raw.npy', activations_np)
np.save(f'{SAVE_DIR}/labels_L20.npy',          labels_np)
np.save(f'{SAVE_DIR}/positions_L20.npy',       positions_np)

n_short = (positions_np < MIN_POSITION).sum()
print(f'\n✅ Saved RAW activations: {activations_np.shape} | dtype: {activations_np.dtype}')
print(f'   Mean vector norm: {np.linalg.norm(activations_np, axis=-1).mean():.2f} (should be >> 1 if raw)')
print(f'   Hallucinated: {labels_np.sum()} | Truthful: {(labels_np==0).sum()}')
print(f'   ⚠️  Samples with last-token position < {MIN_POSITION}: {n_short}/{len(positions_np)}')
print(f'      (short prompts = immature features; report/filter these in analysis)')

HaluEval L20 extraction (raw): 100%|██████████| 200/200 [00:07<00:00, 28.21it/s]


✅ Saved RAW activations: (200, 3584) | dtype: float32
   Mean vector norm: 103.64 (should be >> 1 if raw)
   Hallucinated: 97 | Truthful: 103
   ⚠️  Samples with last-token position < 50: 166/200
      (short prompts = immature features; report/filter these in analysis)


---
## 🌐 SESSION 1B — In-Distribution Validation Set (WildChat + Ultra-FineWeb)

**Goal:** reproduce the in-dist baseline (`fve_nrm ≈ 0.752`, cosine ≈ 0.890) to prove the pipeline is correct.

**Protocol:**
- 50% `allenai/WildChat-1M` (conversational) + 50% `openbmb/Ultra-FineWeb` (web text)
- Sample ONE token per document at random position `>= MIN_POSITION` (50)
- Documents must be long enough: `seq_len > MIN_POSITION`
- Raw vectors, no normalization
- **Document-level dedup:** every sampled doc is tagged with a stable `doc_id` (source ID field if
  present, else a content hash) and we refuse to sample the same `doc_id` twice within this run.

> Uses `streaming=True` so we don't download the full corpora.
> ⚠️ **Leakage caveat:** the `doc_id` dedup above only guarantees no duplicate within *our own*
> sample. Neither `allenai/WildChat-1M` nor `openbmb/Ultra-FineWeb`'s card, nor the `kitft` NLA
> checkpoint repos, publish the `doc_id` list used for the NLA's original training split — so we
> cannot verify zero overlap against *that*. With 1M+ source docs and a 200-doc sample this is
> unlikely to matter much, but it is an unverified assumption, not a guarantee — document it as a
> limitation in the thesis rather than treating it as solved.

In [7]:
import random, hashlib
random.seed(42)

N_INDIST_PER_SOURCE = 100   # 100 WildChat + 100 Ultra-FineWeb = 200 total
MAX_LEN_INDIST      = 1024

def _stable_doc_id(row, text, id_keys):
    """Use the source dataset's own ID field if present; else hash the content.
    Guarantees a doc never gets sampled twice within our own run (see leakage
    caveat above — this does NOT verify non-overlap with the NLA's training split)."""
    for k in id_keys:
        v = row.get(k)
        if v is not None:
            return f'{k}:{v}'
    return 'sha256:' + hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]

def wildchat_texts(n):
    """Yield (doc_id, text) from WildChat-1M (gated — needs HF login)."""
    ds = load_dataset('allenai/WildChat-1M', split='train', streaming=True)
    count = 0
    for row in ds:
        try:
            conv = row['conversation']
            text = '\n'.join(f"{t['role']}: {t['content']}" for t in conv)
        except Exception:
            continue
        if len(text) < 400:      # cheap pre-filter before tokenizing
            continue
        doc_id = _stable_doc_id(row, text, ['conversation_hash', 'conversation_id'])
        yield doc_id, text
        count += 1
        if count >= n * 3:       # over-yield; we filter by token length later
            break

def ultrafineweb_texts(n):
    """Yield (doc_id, text) from Ultra-FineWeb (English config)."""
    ds = load_dataset('openbmb/Ultra-FineWeb', split='en', streaming=True)   # 'en'/'zh' are SPLITS here, not configs
    count = 0
    for row in ds:
        text = row.get('content') or row.get('text') or ''
        if len(text) < 400:
            continue
        doc_id = _stable_doc_id(row, text, ['id', 'doc_id', 'url'])
        yield doc_id, text
        count += 1
        if count >= n * 3:
            break

def extract_indist(doc_text_iter, n_target, desc, seen_doc_ids):
    vecs, poss, doc_ids = [], [], []
    pbar = tqdm(total=n_target, desc=desc)
    for doc_id, text in doc_text_iter:
        if len(vecs) >= n_target:
            break
        if doc_id in seen_doc_ids:
            continue                              # already sampled this doc — skip
        # Tokenize first to check length without running the model
        ids = tokenizer(text, truncation=True, max_length=MAX_LEN_INDIST)['input_ids']
        if len(ids) <= MIN_POSITION + 1:
            continue                              # too short — skip
        pos = random.randint(MIN_POSITION, len(ids) - 1)
        vec, actual_pos, _ = extract_at_position(text, position=pos,
                                                 max_length=MAX_LEN_INDIST)
        vecs.append(vec)
        poss.append(actual_pos)
        doc_ids.append(doc_id)
        seen_doc_ids.add(doc_id)
        pbar.update(1)
    pbar.close()
    return vecs, poss, doc_ids

_seen_doc_ids = set()   # shared across both sources — cross-source collisions are impossible
                        # (different id namespaces) but keeps the guarantee explicit
wc_vecs, wc_pos, wc_doc_ids = extract_indist(wildchat_texts(N_INDIST_PER_SOURCE),
                                             N_INDIST_PER_SOURCE, 'WildChat', _seen_doc_ids)
uf_vecs, uf_pos, uf_doc_ids = extract_indist(ultrafineweb_texts(N_INDIST_PER_SOURCE),
                                             N_INDIST_PER_SOURCE, 'Ultra-FineWeb', _seen_doc_ids)

indist_np = torch.stack(wc_vecs + uf_vecs).numpy().astype(np.float32)
source_np = np.array([0]*len(wc_vecs) + [1]*len(uf_vecs))  # 0=WildChat, 1=UFW
doc_ids_np = np.array(wc_doc_ids + uf_doc_ids, dtype=object)

np.save(f'{SAVE_DIR}/activations_indist_raw.npy', indist_np)
np.save(f'{SAVE_DIR}/sources_indist.npy',         source_np)
np.save(f'{SAVE_DIR}/doc_ids_indist.npy',         doc_ids_np)

print(f'\n✅ In-dist set: {indist_np.shape} | positions all >= {MIN_POSITION} ✅')
print(f'   WildChat: {len(wc_vecs)} | Ultra-FineWeb: {len(uf_vecs)}')
print(f'   Mean vector norm: {np.linalg.norm(indist_np, axis=-1).mean():.2f}')
print(f'   ✅ {len(_seen_doc_ids)} unique doc_ids, 0 repeats within this run')
print(f'   ⚠️  Cannot verify non-overlap against the NLA\'s original training split — see caveat above')

WildChat:   0%|          | 0/100 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Ultra-FineWeb:   0%|          | 0/100 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2048 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/256 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2048 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/256 [00:00<?, ?it/s]

Ultra-FineWeb: 100%|██████████| 100/100 [00:34<00:00,  2.91it/s]


✅ In-dist set: (200, 3584) | positions all >= 50 ✅
   WildChat: 100 | Ultra-FineWeb: 100
   Mean vector norm: 120.52
   ✅ 200 unique doc_ids, 0 repeats within this run
   ⚠️  Cannot verify non-overlap against the NLA's original training split — see caveat above


In [8]:
import gc

hook.remove()
del model
torch.cuda.empty_cache()
gc.collect()

print(f'✅ Qwen freed | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print('\n⚠️  Restart the kernel now before running Session 2!')

✅ Qwen freed | VRAM: 0.0 GB

⚠️  Restart the kernel now before running Session 2!


---
## 💬 SESSION 2 — NLA Verbalization + AR Scoring

### ⚠️ Prerequisites
1. Kernel must be **fresh** (Qwen not loaded)
2. SGLang AV server must be running:
```bash
python -m sglang.launch_server \
    --model /workspace/models/nla-av \
    --port 30000 \
    --mem-fraction-static 0.85 \
    --disable-radix-cache \
    --trust-remote-code \
    --dtype bfloat16
```
Wait for server ready message before continuing.

**Protocol:** verbalization must use **deterministic greedy decoding** (temperature = 0). Check `NLAClient.generate()` signature in `nla_inference.py` — if it accepts sampling params, pass `temperature=0.0` explicitly; the cell below tries this and falls back if unsupported.

In [4]:
print(1+1)

2


In [5]:
import sys, inspect, torch, numpy as np
sys.path.insert(0, '/workspace/natural_language_autoencoders')
from nla_inference import NLAClient, NLACritic

# ─── Load saved RAW activations ─────────────────────
halu_acts = np.load('/workspace/activations_L20_raw.npy')
labels    = np.load('/workspace/labels_L20.npy')
ind_acts  = np.load('/workspace/activations_indist_raw.npy')
print(f'✅ HaluEval: {halu_acts.shape} | In-dist: {ind_acts.shape}')

# ─── Init NLA clients ─────────────────────────────
client = NLAClient(
    checkpoint_dir='/workspace/models/nla-av',
    sglang_url='http://127.0.0.1:30000'
)
critic = NLACritic(
    checkpoint_dir='/workspace/models/nla-ar',
    device='cuda',
    dtype=torch.bfloat16
)

# ─── Greedy decoding wrapper ────────────────────────
# client.generate(activation, prompt, extract_explanation, **sampling) forwards
# arbitrary SGLang sampling_params via a **kwargs catch-all — 'temperature'/'top_p'
# never appear as literal named parameters, so `k in _gen_params` by name alone
# always misses them. Detect the VAR_KEYWORD catch-all instead.
_gen_params = inspect.signature(client.generate).parameters
_accepts_arbitrary_kw = any(
    p.kind == inspect.Parameter.VAR_KEYWORD for p in _gen_params.values()
)
GREEDY_KW = {}
for k, v in [('temperature', 0.0), ('top_p', 1.0)]:
    if k in _gen_params or _accepts_arbitrary_kw:
        GREEDY_KW[k] = v
print(f'Greedy kwargs supported by client.generate: {GREEDY_KW or "NONE — check nla_inference.py / server defaults!"}')

def verbalize(vec):
    return client.generate(vec, **GREEDY_KW)

# ─── Sanity check: single sample ────────────────────
v = halu_acts[0]
explanation = verbalize(v)
mse, cos    = critic.score(explanation, v)
print(f'\n--- Single-sample test (HaluEval idx=0) ---')
print(f'Verbalization:\n{explanation}')
print(f'MSE: {mse:.4f} | Cosine: {cos:.4f}')
# Determinism check (greedy => identical output)
assert verbalize(v) == explanation, '⚠️ Non-deterministic output — greedy decoding NOT active!'
print('✅ Deterministic (greedy) decoding confirmed')

✅ HaluEval: (200, 3584) | In-dist: (200, 3584)
[NLAClient] nla-av: d_model=3584 inj_scale=150.0 embed_scale=1.00 inj_char='㈎'(id=149705)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at /workspace/models/nla-ar and are newly initialized: ['lm_head.weight', 'model.norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[NLACritic] 21 layers  d_model=3584  mse_scale=59.87
Greedy kwargs supported by client.generate: {'temperature': 0.0, 'top_p': 1.0}

--- Single-sample test (HaluEval idx=0) ---
Verbalization:
Structured format with a factual answer pattern ("When was 'Beauty and the Beast' first released?") establishes an informational article format with a defined answer.

The sentence "The earliest known date for Disney's 'Beauty and the Beast' being a religious film is. " signals a factual claim, implying the next tokens will complete with the founding date or distinguishing characteristic of the film's creation.

Final token "film. " closes a clause ("Its earliest creation date and classification type is Disney's 'Beauty and the Beast'. The first one."), strongly expecting "It was created in..." or "The Disney film 'Beauty and the Beast' was established before..." or similar historical context.
MSE: 0.4611 | Cosine: 0.7695
✅ Deterministic (greedy) decoding confirmed


In [6]:
# ── Hotfix rerun: `client`/`critic` are already loaded in this kernel from the
# previous run — reuse them instead of re-running the cell above (which would
# reload NLACritic and double up ~11GB of GPU memory). Only needed this once;
# a fresh kernel just runs the corrected cell above normally. ──
_gen_params = inspect.signature(client.generate).parameters
_accepts_arbitrary_kw = any(
    p.kind == inspect.Parameter.VAR_KEYWORD for p in _gen_params.values()
)
GREEDY_KW = {}
for k, v in [('temperature', 0.0), ('top_p', 1.0)]:
    if k in _gen_params or _accepts_arbitrary_kw:
        GREEDY_KW[k] = v
print(f'Greedy kwargs supported by client.generate: {GREEDY_KW or "NONE — check nla_inference.py / server defaults!"}')

def verbalize(vec):
    return client.generate(vec, **GREEDY_KW)

v = halu_acts[0]
explanation = verbalize(v)
mse, cos    = critic.score(explanation, v)
print(f'\n--- Single-sample test (HaluEval idx=0) ---')
print(f'Verbalization:\n{explanation}')
print(f'MSE: {mse:.4f} | Cosine: {cos:.4f}')
assert verbalize(v) == explanation, '⚠️ Non-deterministic output — greedy decoding NOT active!'
print('✅ Deterministic (greedy) decoding confirmed')

Greedy kwargs supported by client.generate: {'temperature': 0.0, 'top_p': 1.0}

--- Single-sample test (HaluEval idx=0) ---
Verbalization:
Structured format with a factual answer pattern ("When was 'Beauty and the Beast' first released?") establishes an informational article format with a defined answer.

The sentence "The earliest known date for Disney's 'Beauty and the Beast' being a religious film is. " signals a factual claim, implying the next tokens will complete with the founding date or distinguishing characteristic of the film's creation.

Final token "film. " closes a clause ("Its earliest creation date and classification type is Disney's 'Beauty and the Beast'. The first one."), strongly expecting "It was created in..." or "The Disney film 'Beauty and the Beast' was established before..." or similar historical context.
MSE: 0.4611 | Cosine: 0.7695
✅ Deterministic (greedy) decoding confirmed


In [7]:
from tqdm import tqdm

# ─── Full verbalization (AV) — both datasets ──────────
def run_av(acts, name):
    exps = [verbalize(v) for v in tqdm(acts, desc=f'AV: {name}')]
    np.save(f'/workspace/explanations_{name}.npy', np.array(exps, dtype=object))
    print(f'✅ Saved {len(exps)} → /workspace/explanations_{name}.npy')
    return exps

exps_indist = run_av(ind_acts,  'indist')    # ⭐ run in-dist FIRST (validates pipeline)
exps_halu   = run_av(halu_acts, 'halueval')

AV: indist:  46%|████▌     | 91/200 [07:16<09:32,  5.25s/it]

[NLAClient] WARNING: no <explanation> tags. Raw[:200]='<explanation>\nPoetic/technical forum post format with quoted metadata tags and encrypted text fragments suggesting a crypto app UI display, listing health metrics and timestamp anomalies.\n\nThe phrase '


AV: indist: 100%|██████████| 200/200 [15:56<00:00,  4.78s/it]


✅ Saved 200 → /workspace/explanations_indist.npy


AV: halueval: 100%|██████████| 200/200 [15:51<00:00,  4.76s/it]

✅ Saved 200 → /workspace/explanations_halueval.npy


In [8]:
from tqdm import tqdm

# ─── AR Reconstruction + metrics ─────────────────────
# Formula and both footguns confirmed against natural_language_autoencoders/docs/inference.md:
#   fve = 1 - ((pred_n - gold_n) ** 2).mean() / ((gold_n - mu) ** 2).mean()
# There is no separate "paraphrase ceiling" step in the real protocol — an earlier
# draft assumed one existed and it did not; this IS the final fve_nrm, no further scaling.
def evaluate(exps, acts, name, fve_target, cos_target):
    preds = [critic.reconstruct(e) for e in tqdm(exps, desc=f'AR: {name}')]
    preds = torch.stack(preds)

    mse_scale = critic.mse_scale
    gold   = torch.tensor(acts, dtype=torch.float32)
    gold_n = gold  / gold.norm(dim=-1, keepdim=True)  * mse_scale   # footgun 1: must normalize AR output by hand
    pred_n = preds / preds.norm(dim=-1, keepdim=True) * mse_scale

    mu  = gold_n.mean(dim=0)   # footgun 2: do NOT renormalize mu back onto the unit sphere
    fve = 1 - ((pred_n - gold_n)**2).mean() / ((gold_n - mu)**2).mean()
    cos = torch.nn.functional.cosine_similarity(pred_n, gold_n, dim=-1)

    print(f'\n{"="*52}')
    print(f'  [{name}]')
    print(f'  fve_nrm  : {fve.item():.4f}   (target: ~{fve_target})')
    print(f'  Mean cos : {cos.mean().item():.4f}   (target: ~{cos_target})')
    print(f'  Mean MSE : {(2*(1-cos)).mean().item():.4f}')
    print(f'{"="*52}')
    return fve.item(), cos

# ⭐ In-dist first: if this hits ~0.75 / ~0.89, the pipeline is validated
fve_ind,  cos_ind  = evaluate(exps_indist, ind_acts,  'IN-DIST (WildChat+UFW)', 0.752, 0.890)
fve_halu, cos_halu = evaluate(exps_halu,   halu_acts, 'HaluEval QA (OOD)',      '—',   '—')

print('\n[Interpretation]')
print('- In-dist ≈ targets  → pipeline correct; HaluEval gap = genuine distribution shift')
print('- In-dist << targets → check: raw vectors at capture? greedy decoding? pred_n normalized')
print('  by hand (footgun 1)? mu left un-renormalized (footgun 2)? injection_scale=150?')

results = {
    'fve_indist': fve_ind, 'cos_indist': cos_ind.mean().item(),
    'fve_halueval': fve_halu, 'cos_halueval': cos_halu.mean().item(),
}
import json
with open('/workspace/results_v2.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\n✅ Saved → /workspace/results_v2.json')


AR: IN-DIST (WildChat+UFW): 100%|██████████| 200/200 [00:07<00:00, 28.32it/s]



  [IN-DIST (WildChat+UFW)]
  fve_nrm  : 0.7356   (target: ~0.752)
  Mean cos : 0.9055   (target: ~0.89)
  Mean MSE : 0.1891


AR: HaluEval QA (OOD): 100%|██████████| 200/200 [00:07<00:00, 28.15it/s]


  [HaluEval QA (OOD)]
  fve_nrm  : 0.2932   (target: ~—)
  Mean cos : 0.8042   (target: ~—)
  Mean MSE : 0.3915

[Interpretation]
- In-dist ≈ targets  → pipeline correct; HaluEval gap = genuine distribution shift
- In-dist << targets → check: raw vectors at capture? greedy decoding? pred_n normalized
  by hand (footgun 1)? mu left un-renormalized (footgun 2)? injection_scale=150?

✅ Saved → /workspace/results_v2.json


---
## 📋 Next Steps
1. **Semantic theme analysis** — compare AV verbalization language/themes between hallucinated vs.
   truthful HaluEval samples using `explanations_halueval.npy` + `labels_L20.npy`.
2. **Bonus comparison** — re-run the linear probe on raw (non-normalized) layer-20 vectors against
   whatever normalized-vector probe result exists from earlier work.
3. **Doc-level leakage — tracked, not provably solved.** Session 1B records a `doc_id` per
   in-dist sample and refuses to resample it, but we still can't verify non-overlap against the
   NLA's own (unpublished) training split — that remains a documented thesis limitation.
4. **Download results** from `/workspace/` → terminate pod

> Resolved: the "paraphrase ceiling" division that used to be listed here didn't exist in the real
> protocol — confirmed against `natural_language_autoencoders/docs/inference.md`. `evaluate()`'s
> `fve` **is** `fve_nrm` directly, no further scaling needed.

Key saved files:
- `/workspace/activations_L20_raw.npy` — HaluEval raw L20 activations [200, 3584]
- `/workspace/positions_L20.npy` — token positions (flag < 50)
- `/workspace/activations_indist_raw.npy` — in-dist raw vectors [200, 3584]
- `/workspace/sources_indist.npy` — 0 = WildChat, 1 = Ultra-FineWeb
- `/workspace/doc_ids_indist.npy` — per-sample doc_id (source ID field or content hash)
- `/workspace/explanations_indist.npy`, `/workspace/explanations_halueval.npy`
- `/workspace/results_v2.json` — final `fve_nrm` / cosine metrics for both sets